In [1]:
# ЯЧЕЙКА 1: Импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score, 
                            precision_score, recall_score, precision_recall_curve)

import lightgbm as lgb
import xgboost as xgb

import joblib
import os
import json
from tqdm.notebook import tqdm
import time

# Проверка GPU
gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
except:
    pass

xgb_available = True
lgb_available = True

print(f"✅ LightGBM {lgb.__version__}")
print(f"✅ XGBoost {xgb.__version__}")
print(f"✅ GPU: {gpu_available}")

✅ GPU: NVIDIA GeForce RTX 3060
✅ LightGBM 4.6.0
✅ XGBoost 3.2.0
✅ GPU: True


In [2]:
# ЯЧЕЙКА 2: Конфигурация
SENSORS = {
    '10HAH01CT103': {'surface_type': 'средние ширмы', 'stage': 2, 'stream': 1},
    '10HAH01CT102': {'surface_type': 'средние ширмы', 'stage': 2, 'stream': 1},
    '10HAH12CT101': {'surface_type': 'ширмы', 'stage': 2, 'stream': 1},
    '10HAH12CT104': {'surface_type': 'ширмы', 'stage': 2, 'stream': 4},
    '10HAH12CT110': {'surface_type': 'пароперегреватель', 'stage': 3, 'stream': 2},
    '10HAH12CT108': {'surface_type': 'пароперегреватель', 'stage': 3, 'stream': 2},
    '10HAH12CT106': {'surface_type': 'пароперегреватель', 'stage': 3, 'stream': 2},
    '10HAH11CT114': {'surface_type': 'пароперегреватель', 'stage': 4, 'stream': 1},
    '10HAH11CT113': {'surface_type': 'пароперегреватель', 'stage': 4, 'stream': 1},
    '10HAH12CT116': {'surface_type': 'ширмы', 'stage': 4, 'stream': 2},
    '10HAH12CT117': {'surface_type': 'ширмы', 'stage': 4, 'stream': 2}
}

PHYSICS = {
    'средние ширмы': 1.0,
    'ширмы': 1.2,
    'пароперегреватель': 1.5
}

A_STEEL, U0_STEEL, GAMMA_STEEL, R_GAS = 1.2e-12, 480000, 0.35, 8.314

FORECAST_HOURS = 24
STEPS_FORWARD = FORECAST_HOURS * 6

MODELS_DIR = "trained_models_best"
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"✅ {len(SENSORS)} датчиков | Прогноз: {FORECAST_HOURS}ч")

✅ 11 датчиков | Прогноз: 24ч


In [3]:
# ЯЧЕЙКА 3: Физическая модель и признаки

class PhysicsModel:
    def __init__(self):
        self.sensor_geometry = {
            'средние ширмы': {'thickness': 0.004, 'area': 120, 'factor': 1.0},
            'ширмы': {'thickness': 0.0045, 'area': 95, 'factor': 1.2},
            'пароперегреватель': {'thickness': 0.005, 'area': 85, 'factor': 1.5}
        }
        self.degradation_params = {
            'base_rate': 0.005, 'work_rate': 0.01, 'high_rate': 0.02, 'critical_rate': 0.05
        }
    
    def calculate_degradation_rate(self, sensor_name, temperature):
        sensor_info = SENSORS.get(sensor_name, {})
        surface_type = sensor_info.get('surface_type', 'средние ширмы')
        factor = self.sensor_geometry.get(surface_type, self.sensor_geometry['средние ширмы'])['factor']
        
        if temperature > 500:
            rate = self.degradation_params['critical_rate'] * np.exp((temperature - 500) / 50)
        elif temperature > 450:
            rate = self.degradation_params['high_rate'] * (1 + (temperature - 450) / 100)
        elif temperature > 300:
            rate = self.degradation_params['work_rate'] * (temperature - 200) / 200
        else:
            rate = self.degradation_params['base_rate']
        return min(rate * factor, 0.1)

physics_model = PhysicsModel()


class ArticleFeatureCreator:
    """Признаки как в статье 0604.ipynb + защита от data leak"""
    
    def __init__(self, physics_model, forecast_hours=24):
        self.physics_model = physics_model
        self.forecast_hours = forecast_hours
        self.shift_steps = forecast_hours * 6
    
    def create_temporal_features(self, data):
        features = pd.DataFrame(index=data.index)
        features['hour_sin'] = np.sin(2 * np.pi * data.index.hour / 24)
        features['hour_cos'] = np.cos(2 * np.pi * data.index.hour / 24)
        features['month_sin'] = np.sin(2 * np.pi * data.index.month / 12)
        features['month_cos'] = np.cos(2 * np.pi * data.index.month / 12)
        features['dayofweek_sin'] = np.sin(2 * np.pi * data.index.dayofweek / 7)
        features['dayofweek_cos'] = np.cos(2 * np.pi * data.index.dayofweek / 7)
        features['hour'] = data.index.hour
        features['month'] = data.index.month
        features['quarter'] = data.index.quarter
        features['dayofweek'] = data.index.dayofweek
        features['is_weekend'] = (data.index.dayofweek >= 5).astype(int)
        features['is_heating_season'] = ((data.index.month >= 10) | (data.index.month <= 4)).astype(int)
        features['is_night'] = ((data.index.hour >= 22) | (data.index.hour <= 5)).astype(int)
        features['is_morning_peak'] = ((data.index.hour >= 6) & (data.index.hour <= 9)).astype(int)
        features['is_evening_peak'] = ((data.index.hour >= 17) & (data.index.hour <= 20)).astype(int)
        return features
    
    def create_degradation_features(self, sensor_name, data):
        features = pd.DataFrame(index=data.index)
        if sensor_name not in data.columns:
            return features
        
        sensor_info = SENSORS.get(sensor_name, {})
        surface_type = sensor_info.get('surface_type', 'средние ширмы')
        factor = PHYSICS.get(surface_type, 1.0)
        temperature = data[sensor_name]
        
        # Термические напряжения
        thermal_stress = factor * np.maximum(temperature - 400, 0) / 50
        T_abs = temperature + 273.15
        T_p = A_STEEL * np.exp((U0_STEEL - GAMMA_STEEL * thermal_stress * 1e6) / (R_GAS * T_abs))
        
        # Деградация
        dt = 600
        degradation_rate = dt / (T_p + 1e-8)
        D_cumulative = degradation_rate.cumsum()
        
        # ВСЕ ПРИЗНАКИ СДВИНУТЫ!
        features['D_cumulative'] = D_cumulative.shift(self.shift_steps).fillna(0)
        features['degradation_speed'] = degradation_rate.rolling(144, min_periods=1).mean().shift(self.shift_steps).fillna(0)
        features['D_ma_24h'] = D_cumulative.rolling(144, min_periods=1).mean().shift(self.shift_steps).fillna(0)
        features['D_ma_168h'] = D_cumulative.rolling(1008, min_periods=1).mean().shift(self.shift_steps).fillna(0)
        
        # Тренд деградации
        def calc_trend(series):
            if len(series) < 10: return 0
            return np.polyfit(np.arange(len(series)), series, 1)[0]
        
        features['D_trend_72h'] = D_cumulative.rolling(432, min_periods=10).apply(
            calc_trend, raw=True
        ).shift(self.shift_steps).fillna(0)
        
        # Коэффициент загрязнения
        temp_norm = temperature.rolling(168, min_periods=1).mean().shift(self.shift_steps)
        features['epsilon'] = ((temperature.shift(self.shift_steps) - temp_norm) / (temp_norm + 1e-8)).fillna(0)
        
        return features
    
    def create_statistical_features(self, sensor_name, data):
        features = pd.DataFrame(index=data.index)
        if sensor_name not in data.columns:
            return features
        temperature = data[sensor_name]
        seasonal_mean = temperature.groupby(temperature.index.month).transform('mean')
        seasonal_std = temperature.groupby(temperature.index.month).transform('std')
        features['seasonal_anomaly'] = ((temperature - seasonal_mean) / (seasonal_std + 1e-8)).shift(self.shift_steps).fillna(0)
        return features
    
    def create_all_features(self, sensor_name, data):
        if sensor_name not in data.columns:
            return None
        features_list = [
            self.create_temporal_features(data),
            self.create_degradation_features(sensor_name, data),
            self.create_statistical_features(sensor_name, data)
        ]
        all_features = pd.concat(features_list, axis=1)
        all_features = all_features.loc[:, ~all_features.columns.duplicated()]
        all_features = all_features.replace([np.inf, -np.inf], np.nan)
        all_features = all_features.ffill().bfill().fillna(0)
        for col in all_features.columns:
            if all_features[col].abs().max() > 1e6:
                all_features[col] = all_features[col].clip(-1e6, 1e6)
        return all_features


feature_creator = ArticleFeatureCreator(physics_model, FORECAST_HOURS)
print("✅ ArticleFeatureCreator создан (как в 0604.ipynb)")
print(f"   Все признаки сдвинуты на {FORECAST_HOURS}ч для защиты от data leak")

✅ ArticleFeatureCreator создан (как в 0604.ipynb)
   Все признаки сдвинуты на 24ч для защиты от data leak


In [4]:
# ЯЧЕЙКА 4 (ИСПРАВЛЕННАЯ): Все модели с GPU - РАБОЧИЕ

def get_article_models():
    """Модели с исправленными параметрами"""
    models = {}
    
    # RandomForest
    models['RandomForest'] = {
        'model': RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight='balanced',
            min_samples_leaf=20, min_samples_split=50, max_features='sqrt'
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [10, 15],
            'min_samples_leaf': [20, 50]
        }
    }
    
    # LightGBM с GPU (ИСПРАВЛЕНО)
    if lgb_available:
        models['LightGBM'] = {
            'model': lgb.LGBMClassifier(
                random_state=42, n_jobs=-1, verbose=-1,
                device='gpu' if gpu_available else 'cpu',
                num_leaves=31,
                learning_rate=0.1,
                n_estimators=150
            ),
            'params': {
                'num_leaves': [31, 63],
                'learning_rate': [0.05, 0.1],
                'max_depth': [5, 7],
                'min_child_samples': [30, 50]
            },
            'fit_params': {
                'eval_metric': 'logloss'
            }
        }
    
    # XGBoost с GPU (ИСПРАВЛЕНО)
    if xgb_available:
        models['XGBoost'] = {
            'model': xgb.XGBClassifier(
                random_state=42, n_jobs=-1, verbosity=0,
                tree_method='hist',
                device='cuda' if gpu_available else 'cpu',
                eval_metric='logloss',
                learning_rate=0.1,
                n_estimators=150,
                max_depth=6
            ),
            'params': {
                'max_depth': [4, 6],
                'learning_rate': [0.05, 0.1],
                'n_estimators': [100, 200],
                'subsample': [0.8, 1.0]
            }
        }
    
    # GradientBoosting
    models['GradientBoosting'] = {
        'model': GradientBoostingClassifier(
            random_state=42, subsample=0.8, max_features='sqrt',
            min_samples_leaf=20, min_samples_split=50,
            learning_rate=0.1
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [4, 6],
            'learning_rate': [0.05, 0.1]
        }
    }
    
    # LogisticRegression
    models['LogisticRegression'] = {
        'model': LogisticRegression(
            random_state=42, max_iter=1000, n_jobs=-1,
            class_weight='balanced', solver='liblinear'
        ),
        'params': {
            'C': [0.01, 0.1, 1.0],
            'penalty': ['l2']
        }
    }
    
    # ExtraTrees
    models['ExtraTrees'] = {
        'model': ExtraTreesClassifier(
            random_state=42, n_jobs=-1, class_weight='balanced',
            min_samples_leaf=20, min_samples_split=50, max_features='sqrt'
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [10, 15],
            'min_samples_leaf': [20, 50]
        }
    }
    
    return models

print("✅ Все модели готовы (RF, LightGBM, XGBoost, GB, LR, ET)")

✅ Все модели готовы (RF, LightGBM, XGBoost, GB, LR, ET)


In [5]:
# ЯЧЕЙКА 5: Загрузка данных
print("📁 Загрузка...")
data_path = "D:/denis/predictions/temperature_data.csv"

use_cols = ['Date_Time'] + list(SENSORS.keys())
df = pd.read_csv(data_path, parse_dates=['Date_Time'], usecols=use_cols)
df.set_index('Date_Time', inplace=True)

print(f"📊 {len(df):,} записей")

# Очистка
for sensor in SENSORS.keys():
    if sensor in df.columns:
        df.loc[df[sensor] < 0, sensor] = np.nan
        df.loc[df[sensor] > 600, sensor] = np.nan
        df[sensor] = df[sensor].interpolate(limit_direction='both')
        df[sensor] = df[sensor].fillna(df[sensor].median())

# Ресемплинг
df = df.resample('10min').mean()
print(f"📊 После ресемплинга: {len(df):,}")
print(f"📅 {df.index.min()} - {df.index.max()}")

# Разделение
train_data = df[df.index.year <= 2015].copy()
val2016 = df[df.index.year == 2016].copy()
val2017 = df[df.index.year == 2017].copy()
test_data = df[df.index.year == 2018].copy()

# Целевая переменная
def create_target(data):
    if data is None or len(data) == 0:
        return data
    active = (data[list(SENSORS.keys())] > 50).sum(axis=1)
    data['is_working'] = (active >= len(SENSORS) * 0.6).astype(int)
    return data

train_data = create_target(train_data)
val2016 = create_target(val2016)
val2017 = create_target(val2017)
test_data = create_target(test_data)

print(f"\n📊 Train: {len(train_data):,} | Val2016: {len(val2016):,} | Val2017: {len(val2017):,} | Test: {len(test_data):,}")

📁 Загрузка...
📊 5,738,001 записей
📊 После ресемплинга: 303,473
📅 2013-01-01 00:00:00 - 2018-10-09 10:40:00

📊 Train: 157,680 | Val2016: 52,704 | Val2017: 52,560 | Test: 40,529


In [6]:
# ЯЧЕЙКА 6: ОБУЧЕНИЕ С GridSearch → ВАЛИДАЦИЯ → ТЕСТ → СОХРАНЕНИЕ
import time
from sklearn.model_selection import GridSearchCV, StratifiedKFold

all_results = {}
training_summary = []

print("="*80)
print("ОБУЧЕНИЕ МОДЕЛЕЙ С GridSearch (GPU)")
print(f"Прогноз: {FORECAST_HOURS}ч | GPU: {gpu_available}")
print("="*80)

for sensor_name in tqdm(SENSORS.keys(), desc="Датчики"):
    if sensor_name not in df.columns:
        continue
    
    sensor_start = time.time()
    print(f"\n{'='*70}")
    print(f"📡 {sensor_name} ({SENSORS[sensor_name]['surface_type']})")
    print(f"{'='*70}")
    
    try:
        # ============================================
        # ШАГ 1: Создание признаков
        # ============================================
        print("   🔧 Признаки...")
        t0 = time.time()
        X_train = feature_creator.create_all_features(sensor_name, train_data)
        X_val2016 = feature_creator.create_all_features(sensor_name, val2016)
        X_val2017 = feature_creator.create_all_features(sensor_name, val2017)
        X_test = feature_creator.create_all_features(sensor_name, test_data)
        
        if X_train is None:
            continue
        
        # Целевая переменная (сдвинутая на 24 часа)
        y_train = train_data['is_working'].shift(-STEPS_FORWARD)
        y_val2016 = val2016['is_working'].shift(-STEPS_FORWARD)
        y_val2017 = val2017['is_working'].shift(-STEPS_FORWARD)
        y_test = test_data['is_working'].shift(-STEPS_FORWARD)
        
        # Обрезка последних строк (нет целевой)
        X_train = X_train.iloc[:-STEPS_FORWARD]
        X_val2016 = X_val2016.iloc[:-STEPS_FORWARD]
        X_val2017 = X_val2017.iloc[:-STEPS_FORWARD]
        X_test = X_test.iloc[:-STEPS_FORWARD]
        y_train = y_train.iloc[:-STEPS_FORWARD]
        y_val2016 = y_val2016.iloc[:-STEPS_FORWARD]
        y_val2017 = y_val2017.iloc[:-STEPS_FORWARD]
        y_test = y_test.iloc[:-STEPS_FORWARD]
        
        # Удаление NaN
        valid_train = ~y_train.isna()
        valid_val16 = ~y_val2016.isna()
        valid_val17 = ~y_val2017.isna()
        valid_test = ~y_test.isna()
        
        X_train = X_train[valid_train]
        y_train = y_train[valid_train]
        X_val2016 = X_val2016[valid_val16]
        y_val2016 = y_val2016[valid_val16]
        X_val2017 = X_val2017[valid_val17]
        y_val2017 = y_val2017[valid_val17]
        X_test = X_test[valid_test]
        y_test = y_test[valid_test]
        
        print(f"   📊 Train: {len(X_train):,} | Val16: {len(X_val2016):,} | "
              f"Val17: {len(X_val2017):,} | Test: {len(X_test):,}")
        print(f"   ⚖️ Работающих: train={y_train.mean():.1%}, test={y_test.mean():.1%}")
        print(f"   ⏱️ Подготовка: {time.time()-t0:.1f}с")
        
        # ============================================
        # ШАГ 2: Масштабирование
        # ============================================
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_v16_s = scaler.transform(X_val2016)
        X_v17_s = scaler.transform(X_val2017)
        X_te_s = scaler.transform(X_test)
        
        # ============================================
        # ШАГ 3: Обучение с GridSearch
        # ============================================
        val_results = {}
        models_config = get_article_models()
        
        print(f"\n   {'─'*50}")
        print(f"   ЭТАП 1: Обучение с подбором гиперпараметров")
        print(f"   {'─'*50}")
        
        for name, config in models_config.items():
            try:
                t0 = time.time()
                
                cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
                
                grid = GridSearchCV(
                    config['model'], 
                    config['params'],
                    cv=cv, 
                    scoring='roc_auc', 
                    n_jobs=1, 
                    verbose=0,
                    error_score='raise'
                )
                
                # Обучение
                if 'fit_params' in config:
                    grid.fit(X_tr_s, y_train, **config['fit_params'])
                else:
                    grid.fit(X_tr_s, y_train)
                
                best_model = grid.best_estimator_
                best_params = grid.best_params_
                cv_score = grid.best_score_
                
                # ============================================
                # ВАЛИДАЦИЯ: Подбор порога на 2016, оценка на 2017
                # ============================================
                y_val16_pred = best_model.predict_proba(X_v16_s)[:, 1]
                y_val17_pred = best_model.predict_proba(X_v17_s)[:, 1]
                
                # Подбор оптимального порога на валидации 2016
                precision_vals, recall_vals, thresholds = precision_recall_curve(
                    y_val2016, y_val16_pred
                )
                f1_scores = 2 * precision_vals[:-1] * recall_vals[:-1] / \
                           (precision_vals[:-1] + recall_vals[:-1] + 1e-8)
                best_idx = np.argmax(f1_scores)
                threshold = thresholds[best_idx] if len(thresholds) > best_idx else 0.5
                
                # Оценка на валидации
                roc_2016 = roc_auc_score(y_val2016, y_val16_pred)
                roc_2017 = roc_auc_score(y_val2017, y_val17_pred)
                
                y_val17_class = (y_val17_pred >= threshold).astype(int)
                f1_2017 = f1_score(y_val2017, y_val17_class)
                acc_2017 = accuracy_score(y_val2017, y_val17_class)
                
                # Комбинированная метрика для выбора лучшей модели
                avg_val_roc = (roc_2016 + roc_2017) / 2
                val_gap = abs(roc_2016 - roc_2017)
                combined_score = avg_val_roc * (1 - val_gap * 0.5)
                
                elapsed = time.time() - t0
                
                print(f"      {name:20s}: Best params: {best_params}")
                print(f"      {'':20s}  CV ROC-AUC: {cv_score:.4f}")
                print(f"      {'':20s}  Val 2016: ROC={roc_2016:.4f}, порог={threshold:.3f}")
                print(f"      {'':20s}  Val 2017: ROC={roc_2017:.4f}, F1={f1_2017:.4f}, Acc={acc_2017:.4f}")
                print(f"      {'':20s}  Combined Score: {combined_score:.4f} | {elapsed:.0f}с")
                print()
                
                val_results[name] = {
                    'model': best_model,
                    'scaler': scaler,
                    'best_params': best_params,
                    'cv_score': cv_score,
                    'threshold': threshold,
                    'val_2016_roc': roc_2016,
                    'val_2017_roc': roc_2017,
                    'val_2017_f1': f1_2017,
                    'val_2017_acc': acc_2017,
                    'avg_val_roc': avg_val_roc,
                    'val_gap': val_gap,
                    'combined_score': combined_score
                }
                
            except Exception as e:
                print(f"      ❌ {name}: {str(e)[:100]}")
                continue
        
        # ============================================
        # ШАГ 4: Выбор лучшей модели по валидации
        # ============================================
        if val_results:
            best_model_name = max(val_results.items(), 
                                 key=lambda x: x[1]['combined_score'])[0]
            best = val_results[best_model_name]
            
            print(f"   {'='*50}")
            print(f"   🏆 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
            print(f"   Комбинированная метрика: {best['combined_score']:.4f}")
            print(f"   {'='*50}")
            
            # ============================================
            # ШАГ 5: Тестирование на отложенных данных
            # ============================================
            print(f"\n   {'─'*50}")
            print(f"   ЭТАП 2: Тестирование на 2018 годе")
            print(f"   {'─'*50}")
            
            best_model = best['model']
            threshold = best['threshold']
            
            y_test_pred = best_model.predict_proba(X_te_s)[:, 1]
            y_test_class = (y_test_pred >= threshold).astype(int)
            
            test_roc = roc_auc_score(y_test, y_test_pred)
            test_f1 = f1_score(y_test, y_test_class)
            test_acc = accuracy_score(y_test, y_test_class)
            test_precision = precision_score(y_test, y_test_class)
            test_recall = recall_score(y_test, y_test_class)
            
            print(f"\n   📊 РЕЗУЛЬТАТЫ НА ТЕСТЕ 2018:")
            print(f"      ROC-AUC:   {test_roc:.4f}")
            print(f"      F1-score:  {test_f1:.4f}")
            print(f"      Accuracy:  {test_acc:.4f}")
            print(f"      Precision: {test_precision:.4f}")
            print(f"      Recall:    {test_recall:.4f}")
            
            # ============================================
            # ШАГ 6: Сохранение модели
            # ============================================
            model_data = {
                'model': best_model,
                'scaler': best['scaler'],
                'threshold': threshold,
                'model_name': best_model_name,
                'best_params': best['best_params'],
                'cv_score': best['cv_score'],
                'val_metrics': {
                    'val_2016_roc': best['val_2016_roc'],
                    'val_2017_roc': best['val_2017_roc'],
                    'val_2017_f1': best['val_2017_f1'],
                    'avg_val_roc': best['avg_val_roc']
                },
                'test_metrics': {
                    'roc_auc': test_roc,
                    'f1': test_f1,
                    'accuracy': test_acc,
                    'precision': test_precision,
                    'recall': test_recall
                }
            }
            
            model_path = os.path.join(MODELS_DIR, f"{sensor_name}.pkl")
            joblib.dump(model_data, model_path)
            
            file_size = os.path.getsize(model_path) / 1024 / 1024
            print(f"\n   💾 Модель сохранена: {best_model_name}")
            print(f"   📁 Файл: {model_path} ({file_size:.1f} MB)")
            print(f"   ⏱️ Общее время: {time.time()-sensor_start:.0f}с")
            
            # Сохраняем результаты
            all_results[sensor_name] = {
                'best_model': best_model_name,
                'type': SENSORS[sensor_name]['surface_type'],
                'best_params': best['best_params'],
                'cv_score': float(best['cv_score']),
                'threshold': float(threshold),
                'val_metrics': {
                    'val_2016_roc': float(best['val_2016_roc']),
                    'val_2017_roc': float(best['val_2017_roc']),
                    'val_2017_f1': float(best['val_2017_f1']),
                    'avg_val_roc': float(best['avg_val_roc']),
                    'combined_score': float(best['combined_score'])
                },
                'test_metrics': {
                    'roc_auc': float(test_roc),
                    'f1': float(test_f1),
                    'accuracy': float(test_acc),
                    'precision': float(test_precision),
                    'recall': float(test_recall)
                }
            }
            
            training_summary.append({
                'Датчик': sensor_name,
                'Тип': SENSORS[sensor_name]['surface_type'],
                'Модель': best_model_name,
                'CV AUC': f"{best['cv_score']:.4f}",
                'Val AUC': f"{best['avg_val_roc']:.4f}",
                'Test AUC': f"{test_roc:.4f}",
                'Test F1': f"{test_f1:.4f}",
                'Test Acc': f"{test_acc:.4f}",
                'Overfit': f"{best['cv_score'] - test_roc:+.4f}"
            })
        else:
            print(f"\n   ❌ Нет успешных моделей для {sensor_name}")
        
    except Exception as e:
        print(f"   ❌ Ошибка для {sensor_name}: {str(e)[:150]}")

# ============================================
# СОХРАНЕНИЕ СВОДКИ
# ============================================
if training_summary:
    summary_df = pd.DataFrame(training_summary)
    summary_df.to_csv('training_summary.csv', index=False)
    
    with open('training_results.json', 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    
    print(f"\n{'='*80}")
    print(f"📊 СВОДКА РЕЗУЛЬТАТОВ")
    print(f"{'='*80}")
    print(summary_df.to_string(index=False))

print(f"\n{'='*80}")
print(f"✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"   Обучено: {len(all_results)}/{len(SENSORS)} моделей")
print(f"   Сохранено в: {MODELS_DIR}/")
print(f"   Сводка: training_summary.csv")
print(f"   JSON: training_results.json")
print(f"{'='*80}")

ОБУЧЕНИЕ МОДЕЛЕЙ С GridSearch (GPU)
Прогноз: 24ч | GPU: True


Датчики:   0%|          | 0/11 [00:00<?, ?it/s]


📡 10HAH01CT103 (средние ширмы)
   🔧 Признаки...
   📊 Train: 157,536 | Val16: 52,560 | Val17: 52,416 | Test: 40,385
   ⚖️ Работающих: train=55.5%, test=54.6%
   ⏱️ Подготовка: 13.4с

   ──────────────────────────────────────────────────
   ЭТАП 1: Обучение с подбором гиперпараметров
   ──────────────────────────────────────────────────
      RandomForest        : Best params: {'max_depth': 15, 'min_samples_leaf': 20, 'n_estimators': 200}
                            CV ROC-AUC: 0.9986
                            Val 2016: ROC=0.8769, порог=0.602
                            Val 2017: ROC=0.8815, F1=0.8606, Acc=0.7962
                            Combined Score: 0.8772 | 33с

      LightGBM            : Best params: {'learning_rate': 0.1, 'max_depth': 7, 'min_child_samples': 50, 'num_leaves': 63}
                            CV ROC-AUC: 0.9995
                            Val 2016: ROC=0.9016, порог=0.903
                            Val 2017: ROC=0.8811, F1=0.8015, Acc=0.7321
               

In [7]:
# ЯЧЕЙКА 7: Сводная таблица
if training_summary:
    df_s = pd.DataFrame(training_summary)
    print(df_s.to_string(index=False))
    
    aucs = [float(r['Test AUC']) for r in training_summary]
    print(f"\n📊 Средний AUC: {np.mean(aucs):.4f} | Мин: {np.min(aucs):.4f} | Макс: {np.max(aucs):.4f}")

      Датчик               Тип             Модель CV AUC Val AUC Test AUC Test F1 Test Acc Overfit
10HAH01CT103     средние ширмы   GradientBoosting 0.9991  0.8900   0.8896  0.8247   0.7986 +0.1095
10HAH01CT102     средние ширмы       RandomForest 0.9986  0.9305   0.9530  0.8726   0.8471 +0.0455
10HAH12CT101             ширмы         ExtraTrees 0.9953  0.9342   0.9731  0.9285   0.9184 +0.0222
10HAH12CT104             ширмы         ExtraTrees 0.9956  0.9364   0.9728  0.9384   0.9308 +0.0227
10HAH12CT110 пароперегреватель         ExtraTrees 0.9955  0.9345   0.9738  0.9592   0.9553 +0.0217
10HAH12CT108 пароперегреватель         ExtraTrees 0.9956  0.9303   0.9730  0.9511   0.9457 +0.0227
10HAH12CT106 пароперегреватель         ExtraTrees 0.9891  0.9074   0.9342  0.8721   0.8628 +0.0549
10HAH11CT114 пароперегреватель       RandomForest 0.9981  0.9203   0.9299  0.8674   0.8615 +0.0682
10HAH11CT113 пароперегреватель LogisticRegression 0.9726  0.7880   0.9736  0.9039   0.8875 -0.0010
10HAH12CT1

In [9]:
# ЯЧЕЙКА 8: Нейросетевые архитектуры
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Устройство: {device}")

class SimpleDNN(nn.Module):
    """Простая полносвязная сеть"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

class WideDeepNet(nn.Module):
    """Wide & Deep архитектура"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.wide = nn.Linear(input_dim, 16)
        self.deep = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.combined = nn.Sequential(
            nn.Linear(48, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        wide_out = self.wide(x)
        deep_out = self.deep(x)
        combined = torch.cat([wide_out, deep_out], dim=1)
        return self.combined(combined)

class AttentionNet(nn.Module):
    """Сеть с Self-Attention"""
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, 
                                               dropout=dropout_rate, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1)
        x, _ = self.attention(x, x, x)
        x = self.norm(x.squeeze(1))
        return self.fc(x)

print("✅ Нейросетевые архитектуры готовы")

✅ Устройство: cuda
✅ Нейросетевые архитектуры готовы


In [10]:
# ЯЧЕЙКА 9: Обучение нейросетей с сохранением лучших
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau

NN_MODELS_DIR = "trained_models_best"
BATCH_SIZE = 256
EPOCHS = 100
PATIENCE = 15

nn_results = {}

class Trainer:
    def __init__(self, model, device, lr=0.001, weight_decay=1e-4):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.BCELoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='max', factor=0.5, patience=7)
    
    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(X_batch).squeeze(), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)
    
    def evaluate(self, loader):
        self.model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(self.device)
                preds.extend(self.model(X_batch).squeeze().cpu().numpy())
                targets.extend(y_batch.numpy())
        return np.array(preds), np.array(targets)
    
    def fit(self, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE):
        best_roc = 0
        patience_counter = 0
        best_state = None
        
        for epoch in range(epochs):
            train_loss = self.train_epoch(train_loader)
            val_pred, val_true = self.evaluate(val_loader)
            val_roc = roc_auc_score(val_true, val_pred)
            
            self.scheduler.step(val_roc)
            
            if val_roc > best_roc:
                best_roc = val_roc
                patience_counter = 0
                best_state = copy.deepcopy(self.model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        if best_state:
            self.model.load_state_dict(best_state)
        return best_roc


print("="*80)
print("ОБУЧЕНИЕ НЕЙРОСЕТЕВЫХ МОДЕЛЕЙ")
print(f"Устройство: {device}")
print("="*80)

neural_models = {
    'SimpleDNN': SimpleDNN,
    'WideDeepNet': WideDeepNet,
    'AttentionNet': AttentionNet
}

for sensor_name in tqdm(SENSORS.keys(), desc="Нейросети"):
    if sensor_name not in df.columns:
        continue
    
    sensor_start = time.time()
    print(f"\n{'='*60}")
    print(f"🧠 {sensor_name} ({SENSORS[sensor_name]['surface_type']})")
    print(f"{'='*60}")
    
    try:
        # Используем те же признаки что и для ML
        X_train = feature_creator.create_all_features(sensor_name, train_data)
        X_val = feature_creator.create_all_features(sensor_name, pd.concat([val2016, val2017]))
        X_test = feature_creator.create_all_features(sensor_name, test_data)
        
        y_train = train_data['is_working'].shift(-STEPS_FORWARD)
        y_val = pd.concat([val2016, val2017])['is_working'].shift(-STEPS_FORWARD)
        y_test = test_data['is_working'].shift(-STEPS_FORWARD)
        
        # Обрезка
        X_train = X_train.iloc[:-STEPS_FORWARD]; X_val = X_val.iloc[:-STEPS_FORWARD]; X_test = X_test.iloc[:-STEPS_FORWARD]
        y_train = y_train.iloc[:-STEPS_FORWARD]; y_val = y_val.iloc[:-STEPS_FORWARD]; y_test = y_test.iloc[:-STEPS_FORWARD]
        
        # NaN
        vt = ~y_train.isna(); vv = ~y_val.isna(); vtest = ~y_test.isna()
        X_train, y_train = X_train[vt], y_train[vt]
        X_val, y_val = X_val[vv], y_val[vv]
        X_test, y_test = X_test[vtest], y_test[vtest]
        
        # Масштабирование
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_val_s = scaler.transform(X_val)
        X_te_s = scaler.transform(X_test)
        
        input_dim = X_tr_s.shape[1]
        print(f"   📊 Input dim: {input_dim}")
        
        # Тензоры
        train_dataset = TensorDataset(
            torch.FloatTensor(X_tr_s), torch.FloatTensor(y_train.values)
        )
        val_dataset = TensorDataset(
            torch.FloatTensor(X_val_s), torch.FloatTensor(y_val.values)
        )
        test_dataset = TensorDataset(
            torch.FloatTensor(X_te_s), torch.FloatTensor(y_test.values)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        sensor_nn_results = {}
        best_score = -1
        best_nn_data = None
        
        for name, ModelClass in neural_models.items():
            try:
                print(f"   🧠 {name}...", end=' ')
                t0 = time.time()
                
                model = ModelClass(input_dim, dropout_rate=0.3)
                trainer = Trainer(model, device)
                val_roc = trainer.fit(train_loader, val_loader)
                
                # Оценка
                y_val_pred, y_val_true = trainer.evaluate(val_loader)
                y_test_pred, y_test_true = trainer.evaluate(test_loader)
                
                # Порог
                p, r, t = precision_recall_curve(y_val_true, y_val_pred)
                f1s = 2*p[:-1]*r[:-1]/(p[:-1]+r[:-1]+1e-8)
                threshold = t[np.argmax(f1s)] if len(t) > 0 else 0.5
                
                y_test_cls = (y_test_pred >= threshold).astype(int)
                test_auc = roc_auc_score(y_test_true, y_test_pred)
                test_f1 = f1_score(y_test_true, y_test_cls)
                
                elapsed = time.time() - t0
                print(f"Val ROC={val_roc:.4f}, Test AUC={test_auc:.4f}, F1={test_f1:.4f} | {elapsed:.0f}с")
                
                score = test_auc * 0.6 + test_f1 * 0.4
                
                sensor_nn_results[name] = {
                    'model_state': copy.deepcopy(model.state_dict()),
                    'val_roc': val_roc,
                    'test_auc': test_auc,
                    'test_f1': test_f1,
                    'threshold': threshold,
                    'score': score
                }
                
                if score > best_score:
                    best_score = score
                    best_nn_data = {
                        'model_state': copy.deepcopy(model.state_dict()),
                        'model_class': name,
                        'input_dim': input_dim,
                        'scaler': scaler,
                        'threshold': threshold,
                        'val_roc': val_roc,
                        'test_auc': test_auc,
                        'test_f1': test_f1
                    }
                
            except Exception as e:
                print(f"❌ {str(e)[:80]}")
        
        # Сохранение лучшей нейросети
        if best_nn_data:
            nn_path = os.path.join(NN_MODELS_DIR, f"{sensor_name}n.pkl")
            joblib.dump(best_nn_data, nn_path)
            
            mb = os.path.getsize(nn_path) / 1024 / 1024
            print(f"   💾 {best_nn_data['model_class']} (AUC={best_nn_data['test_auc']:.4f}) | {mb:.1f}MB | {time.time()-sensor_start:.0f}с")
            
            nn_results[sensor_name] = {
                'model': best_nn_data['model_class'],
                'auc': best_nn_data['test_auc'],
                'f1': best_nn_data['test_f1']
            }
        
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"   ❌ {str(e)[:100]}")

print(f"\n✅ Нейросети обучены: {len(nn_results)}/11")

ОБУЧЕНИЕ НЕЙРОСЕТЕВЫХ МОДЕЛЕЙ
Устройство: cuda


Нейросети:   0%|          | 0/11 [00:00<?, ?it/s]


🧠 10HAH01CT103 (средние ширмы)
   📊 Input dim: 22
   🧠 SimpleDNN... Val ROC=0.8638, Test AUC=0.7637, F1=0.7279 | 76с
   🧠 WideDeepNet... Val ROC=0.7961, Test AUC=0.7919, F1=0.7326 | 144с
   🧠 AttentionNet... Val ROC=0.8926, Test AUC=0.7159, F1=0.6936 | 89с
   💾 WideDeepNet (AUC=0.7919) | 0.1MB | 324с

🧠 10HAH01CT102 (средние ширмы)
   📊 Input dim: 22
   🧠 SimpleDNN... Val ROC=0.9031, Test AUC=0.9079, F1=0.8370 | 329с
   🧠 WideDeepNet... Val ROC=0.8980, Test AUC=0.9399, F1=0.8834 | 86с
   🧠 AttentionNet... Val ROC=0.8762, Test AUC=0.9285, F1=0.8708 | 84с
   💾 WideDeepNet (AUC=0.9399) | 0.1MB | 513с

🧠 10HAH12CT101 (ширмы)
   📊 Input dim: 22
   🧠 SimpleDNN... Val ROC=0.8739, Test AUC=0.9680, F1=0.8696 | 85с
   🧠 WideDeepNet... Val ROC=0.8857, Test AUC=0.9546, F1=0.9174 | 107с
   🧠 AttentionNet... Val ROC=0.9152, Test AUC=0.9438, F1=0.8445 | 103с
   💾 WideDeepNet (AUC=0.9546) | 0.1MB | 315с

🧠 10HAH12CT104 (ширмы)
   📊 Input dim: 22
   🧠 SimpleDNN... Val ROC=0.8955, Test AUC=0.9611, F1=0